In [7]:
import pandas as pd

from get_market_cap_by_ticker import get_market_cap_by_ticker
from DATA.stock_invest_function import *
from get_fs_data_by_ticker import extract_quarterly_fs_data
from get_hscode_processed_data import get_hscode_processed_data
from get_revenue_export_joined_table import get_revenue_export_joined_table
from sarima_endog_forecast import forecast_endog_with_optional_exog

# 1) DB 접속정보
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

ticker = "A005930"
hs_code = "854232"

df_mc = get_market_cap_by_ticker(db_info, ticker)

df_rev = extract_quarterly_fs_data(
    db_info=db_info,
    table_name="korea_fs_data",          # 실제 테이블명으로 교체
    target_indicator="매출액(천원)",       # 원하는 지표명
    ticker="A000660",                     # 원하는 종목코드
)

# df_exog['monthly_raw']   # 월별 데이터
# df_exog['quarterly']     # 분기 데이터
# df_exog['exog']

df_export = get_hscode_processed_data(db_info, hs_code = hs_code)
df_exog = df_export['quarterly']

combined_df, final_combined_data, forecast_df = get_revenue_export_joined_table(
    df_rev=df_rev,
    df_exog=df_exog,
    join_how="outer",
    fill_exog="ffill"   # 필요시
)

✅ A005930 시가총액 4,075건 조회 완료


In [8]:
combined_df, final_combined_data, forecast_df = get_revenue_export_joined_table(
    df_rev=df_rev,
    df_exog=df_exog,
    join_how="right",
    fill_exog="ffill"   # 필요시
)


In [9]:
from sarima_endog_forecast import forecast_endog_with_optional_exog

# combined_df: Date 인덱스 + ['endog_var','exog_var'] (exog_var는 없어도 됨)
# 예: horizon=4 (분기 4개 또는 월 4개)
out = forecast_endog_with_optional_exog(
    combined_df=final_combined_data,
    horizon=4,          # ⬅️ 예측 기간 설정
    hs_code= None,   # ⬅️ None이면 exog_var 사용 안함
    # seasonal_period=4 # 직접 지정도 가능(미지정시 자동 추론)
)

[메모리] forecast_sarima 실행 전: 460.96 MB
[메모리] find_best_sarima_params 실행 전: 460.96 MB
[메모리] find_best_sarima_params 실행 후: 460.48 MB (변화: -0.48 MB)
[메모리] forecast_sarima 실행 후: 460.48 MB (변화: -0.48 MB)


In [10]:
print("used_exog:", out["used_exog"])      # True/False
print("seasonal_period:", out["seasonal_period"])
print("spec:", out["spec"])                # order/seasonal_order/ic
print("forecast:", out["forecast"])

used_exog: False
seasonal_period: 4
spec: {'order': (2, 1, 2), 'seasonal_order': (1, 1, 1, 4), 'ic_value': 2593.688528351626}
forecast: [2.19864892e+10 2.11553904e+10 1.83661880e+10 1.93224494e+10]


In [11]:
from sarima_endog_forecast import forecast_endog_fill_tail

out2 = forecast_endog_fill_tail(final_combined_data, hs_code="854232", exog_strategy="seasonal")
print("used_exog:", out2["used_exog"])
print("spec:", out2["spec"])
print("forecast:", out2["forecast"])
filled_with_exog = out2["filled"]

[메모리] find_best_sarima_params 실행 전: 460.48 MB
[메모리] find_best_sarima_params 실행 후: 460.48 MB (변화: +0.00 MB)
used_exog: True
spec: {'order': (2, 1, 2), 'seasonal_order': (1, 1, 1, 4), 'ic_value': 2543.179763062745}
forecast: [2.18211416e+10 2.42639396e+10 2.27463692e+10 2.77950612e+10]
